# Semantic Study Search в Google Colab

Запуск поиска по загруженным PDF, TXT и Markdown с сохранением библиотеки в Google Drive.

1. В меню **Среда выполнения → Сменить среду выполнения** выберите **Python 3** и **CPU**.
2. Выполните ячейки **1–4** по очереди кнопкой ▶ слева от кода. При подключении Drive разрешите доступ к своему аккаунту.
3. После работы выполните ячейку **5**, включив флажок сохранения, и дождитесь сообщения об успешном сохранении.
4. В новом сеансе снова выполните ячейки **1–4**: библиотека восстановится из последнего архива.

Исходный код загружается из [репозитория проекта](https://github.com/AynurAkhmadullin/semantic-study-search). Перед использованием новых изменений отправьте их в GitHub самостоятельно. Внутри Colab приложение запускается без Docker; зависимости и Python 3.13 устанавливаются через uv.

**Сохранение ручное.** Рабочие файлы находятся на временном диске Colab. Google Drive хранит архивы библиотеки: оригиналы, индекс и состояния загрузок. При внезапном отключении среды изменения после последнего сохранения будут потеряны. Кэш модели в архив не входит; в новом сеансе модель скачивается заново.

Colab предоставляет временную среду, а не постоянный адрес приложения. Учитывайте [ограничения выбранного тарифа Colab](https://research.google.com/colaboratory/faq.html), в том числе на использование веб-интерфейсов в управляемых средах.


## 1. Установка

Дождитесь окончания установки. Повторное выполнение использует существующую копию репозитория и не перезаписывает её. Для получения новой версии из GitHub удобнее начать новый сеанс после сохранения библиотеки.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/content/semantic-study-search")
APP_PYTHON = PROJECT / ".venv/bin/python"

if not Path("/content").is_dir():
    raise RuntimeError("Откройте этот блокнот в Google Colab.")
if globals().get("app") is not None and globals()["app"].poll() is None:
    raise RuntimeError("Перед повторной установкой выполните шаг 5: сохранение и остановку.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv==0.11.29"],
    check=True,
)
if not PROJECT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/AynurAkhmadullin/semantic-study-search.git",
            str(PROJECT),
        ],
        check=True,
    )
elif not (PROJECT / "pyproject.toml").is_file():
    raise RuntimeError(f"В каталоге {PROJECT} нет проекта.")

os.chdir(PROJECT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "sync",
        "--locked",
        "--all-extras",
        "--no-dev",
        "--python",
        "3.13",
    ],
    cwd=PROJECT,
    check=True,
)
print("Установка завершена. Выполните шаг 2.")

## 2. Google Drive и восстановление библиотеки

Архивы сохраняются в **Мой диск → semantic-study-search** с именами `library-*.zip`. Используется та же папка и формат, что и в прежних ячейках запуска.

При первом запуске создаётся пустая библиотека. В новой среде восстанавливается последний завершённый архив. Повторное выполнение этой ячейки сохраняет уже существующую локальную библиотеку. Не используйте одну папку архивов одновременно в нескольких сеансах Colab.


In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from google.colab import drive

LIBRARY = Path("/content/sss-library")
BACKUPS = Path("/content/drive/MyDrive/semantic-study-search")


def restore_library(library: Path, backups: Path) -> str:
    """Создаёт библиотеку или восстанавливает последний архив.

    Args:
        library: Локальная папка библиотеки.
        backups: Папка архивов Google Drive.

    Returns:
        Сообщение о результате восстановления.
    """
    if library.exists() and any(library.iterdir()):
        return "Используется текущая локальная библиотека."

    archives = sorted(backups.glob("library-*.zip"))
    if not archives:
        library.mkdir(parents=True, exist_ok=True)
        return "Создана пустая библиотека."

    latest = archives[-1]
    # Распаковка в отдельную папку не оставит полубиблиотеку при ошибке.
    with TemporaryDirectory(prefix="sss-restore-", dir=library.parent) as temporary:
        restored = Path(temporary) / "library"
        restored.mkdir()
        with ZipFile(latest) as archive:
            if archive.testzip() is not None:
                raise RuntimeError(f"Архив повреждён: {latest.name}")
            archive.extractall(restored)
        if library.exists():
            library.rmdir()
        restored.rename(library)
    return f"Восстановлена библиотека: {latest.name}"


if globals().get("app") is not None and globals()["app"].poll() is None:
    raise RuntimeError("Приложение работает. Для продолжения используйте шаг 4.")

drive.mount("/content/drive")
if not BACKUPS.parent.is_dir():
    raise RuntimeError("Google Drive не подключён. Повторите шаг 2.")
BACKUPS.mkdir(parents=True, exist_ok=True)
print(restore_library(LIBRARY, BACKUPS))

## 3. Запуск API и интерфейса

Первый запуск скачивает модель и может занять несколько минут. Дождитесь сообщения **«Приложение готово»**.

Исправление `XSRF token missing or invalid` записывается непосредственно в `.streamlit/config.toml` копии проекта **в Colab**: `xsrfCookieSameSite = "none"`. Streamlit автоматически добавляет `Secure`; браузер обращается к приложению через HTTPS-прокси Colab. Проверка XSRF и CORS остаётся включённой. Подробнее — в [документации Streamlit](https://docs.streamlit.io/develop/api-reference/configuration/config.toml).

Ячейка проверяет cookie работающего сервера, а не только содержимое файла. Ожидаемый результат: **SameSite: None; Secure: True**. Значение токена не выводится. Повторный запуск останавливает прежний процесс; перед этим дождитесь завершения обработки всех файлов.


In [ ]:
import re
import socket
import time
from http.cookies import SimpleCookie
from urllib.parse import urlsplit

import requests
from google.colab import output

LOG_PATH = Path("/content/sss.log")


def configure_streamlit(config_path: Path) -> None:
    """Настраивает XSRF-cookie для интерфейса внутри Colab.

    Args:
        config_path: Конфигурация копии проекта в Colab.
    """
    config_text = config_path.read_text(encoding="utf-8")
    # Удаляем старые значения, чтобы повторный запуск не создавал дубликаты.
    for key in ("xsrfCookieSameSite", "enableXsrfProtection", "enableCORS"):
        config_text = re.sub(rf"(?m)^[ \t]*{key}[ \t]*=.*\n?", "", config_text)
    settings = (
        '[server]\nxsrfCookieSameSite = "none"\nenableXsrfProtection = true\nenableCORS = true'
    )
    if "[server]" in config_text:
        config_text = config_text.replace("[server]", settings, 1)
    else:
        config_text = settings + "\n\n" + config_text
    config_path.write_text(config_text, encoding="utf-8")


def stop_app() -> None:
    """Останавливает запущенный блокнотом процесс и его сервисы."""
    if "app" in globals() and app.poll() is None:
        app.terminate()
        app.wait(timeout=45)


def check_ports_free() -> None:
    """Проверяет, что API и интерфейс полностью остановлены."""
    for port in (8000, 8501):
        with socket.socket() as probe:
            try:
                probe.bind(("127.0.0.1", port))
            except OSError as error:
                raise RuntimeError(
                    f"Порт {port} занят. Проверьте предыдущий запуск приложения. "
                    "Не удаляйте среду до сохранения библиотеки."
                ) from error


if not LIBRARY.is_dir() or not APP_PYTHON.is_file():
    raise RuntimeError("Сначала выполните шаги 1 и 2.")

stop_app()
check_ports_free()
configure_streamlit(PROJECT / ".streamlit/config.toml")

proxy_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
proxy = urlsplit(proxy_url)
if proxy.scheme != "https" or not proxy.hostname:
    raise RuntimeError("Colab не вернул HTTPS-адрес. Обновите страницу и повторите шаг 3.")

app_env = os.environ.copy()
# Настройка берётся из файла даже после прежних экспериментов с окружением.
for variable in (
    "STREAMLIT_SERVER_XSRF_COOKIE_SAME_SITE",
    "STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION",
    "STREAMLIT_SERVER_ENABLE_CORS",
):
    app_env.pop(variable, None)
app_env.update(
    SSS_INDEX_DIR=str(LIBRARY),
    SSS_UPLOAD_DIR=str(LIBRARY / "uploads"),
    SSS_MODEL_CACHE="/content/sss-model-cache",
    SSS_OFFLINE="0",
    STREAMLIT_BROWSER_SERVER_ADDRESS=proxy.hostname,
    STREAMLIT_BROWSER_SERVER_PORT="443",
)

print("Запускаем приложение и загружаем модель…")
with LOG_PATH.open("w", encoding="utf-8") as log:
    app = subprocess.Popen(
        [str(APP_PYTHON), "-u", "-m", "semantic_study_search.launcher"],
        cwd=PROJECT,
        env=app_env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

try:
    with requests.Session() as session:
        session.trust_env = False
        deadline = time.monotonic() + 900
        next_report = time.monotonic() + 30
        while time.monotonic() < deadline:
            if app.poll() is not None:
                raise RuntimeError("Приложение завершилось при запуске.")
            try:
                response = session.get("http://127.0.0.1:8501/_stcore/health", timeout=2)
                response.raise_for_status()
                api_response = session.get("http://127.0.0.1:8000/health", timeout=2)
                api_response.raise_for_status()
            except requests.RequestException:
                if time.monotonic() >= next_report:
                    print("Ожидаем готовности. Загрузка модели может занять несколько минут…")
                    next_report = time.monotonic() + 30
                time.sleep(1)
            else:
                break
        else:
            raise RuntimeError("Истекло время ожидания запуска (15 минут).")

    cookies = SimpleCookie()
    cookies.load(response.headers.get("Set-Cookie", ""))
    cookie = cookies.get("_streamlit_xsrf")
    same_site = cookie["samesite"] if cookie else ""
    secure = bool(cookie["secure"]) if cookie else False
    print("SameSite:", same_site or "cookie отсутствует")
    print("Secure:", secure)
    if same_site.lower() != "none" or not secure:
        raise RuntimeError("Настройка XSRF не применилась. Проверьте вывод диагностики.")
except BaseException:
    stop_app()
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise

print("Приложение готово. Выполните шаг 4.")

## 4. Открытие интерфейса

Выберите файл в «Добавить материал», дождитесь окончания его передачи и нажмите «Загрузить материал». Дождитесь готовности материала к поиску.

После повторного запуска шага 3 выполните эту ячейку ещё раз, чтобы открыть свежий интерфейс.

Если загрузка снова вернула **403**, проверьте успешный вывод шага 3 и разрешите сторонние cookie для страницы Colab в браузере. Прокси приложения может использовать домен `*.colab.dev`; разрешения только для `googleusercontent.com` недостаточно. После изменения настроек браузера обновите страницу блокнота и снова выполните эту ячейку.


In [ ]:
if "app" not in globals() or app.poll() is not None:
    raise RuntimeError("Сначала запустите приложение: шаг 3.")

output.serve_kernel_port_as_iframe(8501, height=850)

## 5. Сохранение библиотеки и остановка — после работы

Дождитесь завершения обработки файлов и не загружайте новые во время сохранения. **Включите флажок `SAVE_AND_STOP` ниже** (или замените `False` на `True` в коде) и выполните ячейку.

Приложение остановится до создания архива. Новый архив проверяется после копирования в Google Drive и только затем получает имя `library-*.zip`. Незавершённые `.part` не используются при восстановлении. Предыдущие архивы сохраняются.

Дождитесь сообщения **«Библиотека сохранена в Google Drive»**, затем можно отключить среду. Для продолжения работы в той же среде снова выполните шаги 3–4. Это ручное сохранение: закрытие вкладки само по себе его не выполняет.

По умолчанию флажок выключен, чтобы первоначальная команда «Выполнить все» не остановила только что запущенное приложение.


In [ ]:
import shutil
from datetime import UTC, datetime


def save_library(library: Path, backups: Path) -> Path:
    """Сохраняет архив остановленной библиотеки и проверяет копию.

    Args:
        library: Локальная папка библиотеки.
        backups: Подключённая папка Google Drive.

    Returns:
        Путь завершённого архива в Google Drive.
    """
    if not library.is_dir() or not backups.is_dir():
        raise RuntimeError("Нет библиотеки или папки Google Drive. Проверьте шаг 2.")

    stamp = datetime.now(UTC).strftime("%Y%m%d-%H%M%S-%f")
    destination = backups / f"library-{stamp}.zip"
    incomplete = destination.with_suffix(".zip.part")
    with TemporaryDirectory(prefix="sss-backup-", dir=library.parent) as temporary:
        archive_path = shutil.make_archive(
            str(Path(temporary) / "library"), "zip", root_dir=library
        )
        shutil.copy2(archive_path, incomplete)
        with ZipFile(incomplete) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("Копия повреждена. Повторите сохранение.")
        incomplete.rename(destination)
    return destination


SAVE_AND_STOP = False  # @param {type:"boolean"}

if SAVE_AND_STOP:
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive не подключён. Подключите его перед сохранением.")
    if "app" in globals() and app.poll() is None:
        with requests.Session() as session:
            session.trust_env = False
            response = session.get("http://127.0.0.1:8000/materials", timeout=10)
            response.raise_for_status()
            active = [
                item
                for item in response.json()["materials"]
                if item["status"] not in {"ready", "failed"}
            ]
        if active:
            raise RuntimeError("Ещё обрабатываются файлы. Дождитесь их готовности и повторите.")
    stop_app()
    check_ports_free()
    saved = save_library(LIBRARY, BACKUPS)
    print("Библиотека сохранена в Google Drive:", saved.name)
    print("Папка: Мой диск → semantic-study-search")
    print("Приложение остановлено. Теперь можно отключить среду Colab.")
else:
    print("После работы включите SAVE_AND_STOP и выполните эту ячейку для сохранения.")

## Диагностика — только при ошибке

Включите `RUN_DIAGNOSTICS` и выполните ячейку. Она показывает готовность локальных сервисов, атрибуты XSRF-cookie и последние строки журнала; значения cookie не выводятся.

Если шаг 3 показывает **SameSite: None** и **Secure: True**, но файл не передаётся до нажатия «Загрузить материал», проверьте браузерный запрос `upload_file` во вкладке Network. Передача в Streamlit происходит до обработки PDF, построения индекса и сохранения архива в Drive.

Не публикуйте значения cookie и токены из DevTools.


In [ ]:
RUN_DIAGNOSTICS = False  # @param {type:"boolean"}

if RUN_DIAGNOSTICS:
    with requests.Session() as session:
        session.trust_env = False
        for label, url in (
            ("API", "http://127.0.0.1:8000/health"),
            ("Streamlit", "http://127.0.0.1:8501/_stcore/health"),
        ):
            try:
                response = session.get(url, timeout=5)
                print(label, "HTTP:", response.status_code)
                if label == "Streamlit":
                    cookies = SimpleCookie()
                    cookies.load(response.headers.get("Set-Cookie", ""))
                    cookie = cookies.get("_streamlit_xsrf")
                    print("Сервер выдаёт XSRF-cookie:", cookie is not None)
                    if cookie is not None:
                        print("SameSite:", cookie["samesite"])
                        print("Secure:", bool(cookie["secure"]))
                else:
                    print(response.text)
            except requests.RequestException as error:
                print(label, type(error).__name__)

    if LOG_PATH.is_file():
        print("\nПоследние строки журнала:")
        print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])